In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

##### BOOKINGS

@dlt.table(name="silver.stage_bookings")
def stage_bookings():
    df = spark.readStream.format("delta") \
        .load("/Volumes/workspace/bronze/bronzevolume/bookings/data")
    return df 

@dlt.view(name="trans_bookings")
def trans_bookings():
    df = spark.readStream.table("silver.stage_bookings")
    df = df.withColumn("amount", col("amount").cast("double")) \
      .withColumn("modifyDate", current_timestamp()) \
      .withColumn("booking_date", to_date(col("booking_date"))) \
      .drop("_rescued_data")
    return df  

@dlt.table(name="silver.silver_bookings")
@dlt.expect_all_or_drop({
    "rule1": "booking_id IS NOT NULL",
    "rule2": "passenger_id IS NOT NULL"
})
def silver_bookings():
    df = spark.readStream.table("trans_bookings")
    return df 

##### FLIGHTS

@dlt.view(name="trans_flights")
def trans_flights():
    df = spark.readStream.format("delta") \
        .load("/Volumes/workspace/bronze/bronzevolume/flights/data/")
    df = df.drop("_rescued_data")\
        .withColumn("modifiedDate",current_timestamp())
    return df

@dlt.table(name="silver.silver_flights")
def silver_flights():
    df = spark.readStream.table("trans_flights")
    return df

##### PASSENGERS

@dlt.view(name="trans_passengers")
def trans_passengers():
    df = spark.readStream.format("delta") \
        .load("/Volumes/workspace/bronze/bronzevolume/customers/data/")
    df = df.drop("_rescued_data")\
        .withColumn("modifiedDate",current_timestamp())
    return df

@dlt.table(name="silver.silver_passengers")
def silver_passengers():
    df = spark.readStream.table("trans_passengers")
    return df

##### AIRPORTS

@dlt.view(name="trans_airports")
def trans_airports():
    df = spark.readStream.format("delta") \
        .load("/Volumes/workspace/bronze/bronzevolume/airports/data/")
    df = df.drop("_rescued_data")\
        .withColumn("modifiedDate",current_timestamp())
    return df

@dlt.table(name="silver.silver_airports")
def silver_airports():
    df = spark.readStream.table("trans_airports")
    return df

#### Silver Business View 

@dlt.table(
    name = "silver.silver_business"
)
def silver_business():
    df = (dlt.readStream("silver.silver_bookings")
          .join(dlt.readStream("silver.silver_flights"), ["flight_id"])
          .join(dlt.readStream("silver.silver_passengers"), ["passenger_id"])
          .join(dlt.readStream("silver.silver_airports"), ["airport_id"])
          .drop("modifiedDate"))
    return df